# 06. California carrot test supplement

Collects an independent California DWR 2023 carrot pilot. The reference query uses explicit T6 single-crop fields whose reported main-crop day falls within May–August. The window still needs imagery and season checks.

Select the project Python kernel and click **Run All**. Rerun with the same settings to resume pending work. This collection has its own test campaign. Primary JECAM and standard preprocessing continue to use their existing inputs.

## 1. Setup and configuration

Choose a Python 3.12+ kernel, review the settings below, then **Restart Kernel → Run All**. Opening the notebook starts no work. Rerun with the same settings to resume saved batches and jobs. Set `run_satellite_collection = False` for preparation without satellite calls; preparation can still save selections.


In [ ]:
# Review these settings before Run All.
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import notebook_setup
from crop_pipeline.preprocessing_workflow import paths
from crop_pipeline.sources import adaptive, annual, eurocrops, supplemental

run_satellite_collection = True
submit_only = False
download_missing_references = True
parcels_per_batch = 50
extra_batches = 1  # Additional slots after the initial pilot batch.
settings_path = root / "settings.local.json"
settings = notebook_setup.ensure_local_settings(settings_path)
_, report, _ = paths(root)
archive = annual.archive_root(settings, report, repository_root=root)

## 2. Obtain references

Reuse the saved California DWR 2023 reference snapshot. If it is missing, `download_missing_references = True` allows the bounded single-crop carrot query when the next cell loads and validates references. Set it to `False` to require a local snapshot, independently of satellite collection.


In [ ]:
references = root / "data/external/california_dwr/carrot_2023.geojson"
print("Reference file:", references)


## 3. Validate references

Load and validate the reference snapshot. Retain explicit T6 single-crop fields with reported main-crop day 121–243, valid polygon geometry and unique field identities. These dates do not replace imagery and season checks.


In [ ]:
features = supplemental.load_california(references, download_if_missing=download_missing_references)
print("Eligible declared references before overlap screening:", len(features))


## 4. Prepare the collection round

Screen candidates against existing development and reserved CropHarvest test footprints. Resume saved plans and select unused parcels within the existing batch limits. Preserve the separate supplemental test campaign and locked partitions.


In [ ]:
batches = supplemental.prepare(
    "california_dwr", report, archive, features, references,
    limit=parcels_per_batch, extra_batches=extra_batches,
)


## 5. Collect annual imagery

Use `submit_only = True` to queue exports or `False` to download finished results. Keep the laptop on until Run All returns normally. Submitted jobs continue in the cloud; rerun with the same settings to retrieve results and create annual TIFFs.


In [ ]:
project = notebook_setup.project_for_collection(settings, settings_path, required=run_satellite_collection)
result = adaptive.collect_batches(
    batches, project, enabled=run_satellite_collection, authenticate=run_satellite_collection,
    submit_only=submit_only, download_workers=2, intervals_per_request=6,
)


## 6. Summary and next action

Check collection status below. If jobs remain pending, rerun with the same settings and `submit_only = False` to retrieve completed exports. Use collection 08 for a saved progress snapshot. Assess supplemental imagery quality and usable windows separately; standard preprocessing reports primary JECAM and registered supplements separately.


In [ ]:
print("Collection status:", result["status"])


## Organized outputs

Refresh the linked output folders without moving downloads or changing saved jobs. Run **08_dataset_summary.ipynb** to record saved batches and selections. Supplemental test campaigns remain separate from primary JECAM.


In [ ]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(root)
print("Organized collection outputs:", root / "pipeline_outputs/collection")
print("Next: data_collection/08_dataset_summary.ipynb; assess supplemental usable windows separately.")


## Evidence and interpretation

Counts describe selected references and collection progress. Imagery and window checks determine usable samples. See [research basis and limits](../../docs/research_basis.md) for sources and project-specific choices.
